# Run DIODE on a small synthetic dataset

This notebook shows the minimal data files and config needed to run DIODE on user data. The ground-truth matrix is generated only so the example can compute an AUROC after fitting.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from scipy.linalg import expm

from diode import DIODE
from diode.experiment import DIODEBenchmark

## Create example inputs

In this section we simulate the files a user would normally already have. For a real dataset, replace `datamatrix`, `pseudotime`, and `gene_names` with your measured expression matrix, ordered pseudotime values, and gene list.

### Required input files

#### Format of the expression file

The expression file is a `G x C` matrix of expression values, where `G` is the number of genes and `C` is the number of cells or timepoints. Each row corresponds to one gene, and each column corresponds to one cell/timepoint. The file can be whitespace-separated text.

#### Example expression file

```text
1.24  1.21  1.28  ...
0.00  0.19  0.00  ...
...
```

#### Format of the pseudotime file

The pseudotime file contains one value per cell/timepoint. Its length must match the number of columns in the expression matrix. The runner sorts the expression columns by these pseudotime values before fitting.

#### Example pseudotime file

```text
0.000
0.034
0.069
...
1.000
```

#### Format of the gene file

The gene file contains one gene name per row. Its length must match the number of rows in the expression matrix, and this order defines the row/column order used for learned and reference matrices.

#### Optional reference network

A reference network is only needed if you want AUROC/AUPR evaluation. It should be a square `G x G` matrix with rows as target genes and columns as source/regulator genes.

The optional `A_true` matrix below is only used because this is a tutorial dataset and we want to compute AUROC at the end.

In [3]:
rng = np.random.default_rng(1004)

n_genes = 10
n_timepoints = 50
gene_names = [f"Gene{i + 1}" for i in range(n_genes)]
pseudotime = np.linspace(0.0, 1.0, n_timepoints)

# Ground truth used only for this tutorial.
# A_true[target, regulator] is the effect of a regulator gene on a target gene.
# We draw random signed weights and keep edges whose absolute weight is larger than 0.6.
# In a real dataset, the true network would be unknown unless you have a reference network.
A_random = rng.uniform(-1.0, 1.0, size=(n_genes, n_genes))
np.fill_diagonal(A_random, 0.0)
edge_threshold = 0.6
A_true = np.where(np.abs(A_random) > edge_threshold, A_random, 0.0)

# Negative diagonal terms keep the simulated linear system stable by pushing eigenvalues left.
# If the eigenvalues are too close to zero, the trajectory can become too flat to identify edges.
np.fill_diagonal(A_true, -0.7)

# Binary ground truth for AUROC evaluation. Self-decay terms on the diagonal are not regulatory edges.
A_true_binary = (A_true != 0).astype(int)
np.fill_diagonal(A_true_binary, 0)

x0 = rng.normal(loc=1.0, scale=0.2, size=n_genes)
X_cells_by_genes = np.vstack([expm(A_true * t) @ x0 for t in pseudotime])
X_cells_by_genes += rng.normal(loc=0.0, scale=0.02, size=X_cells_by_genes.shape)
datamatrix = X_cells_by_genes.T

print("A_true shape:", A_true.shape)
print("datamatrix shape:", datamatrix.shape)
print("pseudotime shape:", pseudotime.shape)


A_true shape: (10, 10)
datamatrix shape: (10, 50)
pseudotime shape: (50,)


## Save the input files

The package expects a genes-by-cells/timepoints expression matrix. In your own analysis, these files should point to your real data. The reference-network files are included here only because this synthetic example has a known ground truth.

In [4]:
example_dir = Path("example_data")
example_dir.mkdir(exist_ok=True)

np.savetxt(example_dir / "datamatrix.txt", datamatrix)
np.savetxt(example_dir / "pseudotime.txt", pseudotime)
np.savetxt(example_dir / "genes.txt", np.asarray(gene_names), fmt="%s")

pd.DataFrame(A_true_binary, index=gene_names, columns=gene_names).to_csv(example_dir / "reference_network.csv")
pd.DataFrame(np.sign(A_true).astype(int), index=gene_names, columns=gene_names).to_csv(example_dir / "reference_network_signed.csv")


## Write the config

The saved YAML uses the same keys as the benchmark configs, including the training and cv sections. For the tutorial, the user-facing choices are the huber_delta and l2_lambda grids; the rest of the training values are standard defaults filled in before saving the config.

This synthetic dataset has 50 timepoints, but the CV settings are still kept small so the tutorial runs quickly.

In [5]:
hyperparameter_grid = {
    "huber_delta": [0.01, 2.0],
    "l2_lambda": [1e-7, 1e-5],
}

training_defaults = {
    "loss_type": "huber",
    "huber_delta": hyperparameter_grid["huber_delta"][0],
    "l1_lambda": 0.0,
    "l2_lambda": hyperparameter_grid["l2_lambda"][0],
    "lr": 1.0,
    "epochs": 1001,
    "print_every": 500,
    "lbfgs_max_iter": 20,
    "lbfgs_history_size": 10,
    "lbfgs_line_search_fn": "strong_wolfe",
    "optimizer_tolerance_grad": 0.001,
    "stop_tolerance_grad": 0.001,
    "stop_tolerance_change": 1e-5,
    "stop_on_loss": True,
    "stop_on_parameter": True,
    "stop_on_optimizer_step_size": False,
    "stop_on_grad": False,
    "patience_level": 5,
    "dtype": "float64",
    "a_nnz_threshold": 1e-6,
}

config = {
    "dataset": "example_data",
    "data_dir": str(example_dir),
    "reference_network_dir": str(example_dir),
    "output_dir": "outputs/example_data",
    "expression": str(example_dir / "datamatrix.txt"),
    "pseudotime": str(example_dir / "pseudotime.txt"),
    "genes": str(example_dir / "genes.txt"),
    "seed": 0,
    "device": "auto",
    "cv": {"n_splits": 3, "n_bins": 3, "seed": 0},
    "training": training_defaults,
    "hyperparameter_grid": {
        **hyperparameter_grid,
        "tolerance_change": [training_defaults["stop_tolerance_change"]],
    },
}

config_path = example_dir / "config.yaml"
with config_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(config, handle, sort_keys=False)

print(config_path)


example_data/config.yaml


## Fit DIODE

The `fit()` call runs the same workflow used by the dataset runner: cross-validation over the hyperparameter grid, selection by validation error and rank stability, final refit, and perturbation scoring.

In [6]:
model = DIODE(config_path)
result = model.fit()

print("Output directory:", result["output_dir"])
print("Selected hyperparameters:")
print({
    "huber_delta": result["metrics"]["huber_delta"],
    "l2_lambda": result["metrics"]["l2_lambda"],
})


[001/004] hd_0.01_l2_1e-07_tc_1e-05
initial train_total=3.48884251e-03 train_data=3.48883251e-03 val_data=3.46026372e-03
epoch 0000 train=3.48883251e-03 val=3.46026372e-03 grad_max=7.709e-04 step_max=nan loss_change=nan dA=0.000e+00 dX*=0.000e+00 best_val=3.46026372e-03@0 warning=internal LBFGS gradient tolerance was already met; this epoch may have had no effective parameter update
epoch 0004 train=3.48883251e-03 val=3.46026372e-03 grad_max=7.709e-04 step_max=nan loss_change=nan dA=0.000e+00 dX*=0.000e+00 best_val=3.46026372e-03@0 stop=parameter_change (delta_A_max_abs and delta_X_star_max_abs) <= stop_tolerance_change for 5 consecutive epochs warning=internal LBFGS gradient tolerance was already met; this epoch may have had no effective parameter update
initial train_total=3.66806902e-03 train_data=3.66805902e-03 val_data=3.46026372e-03
epoch 0000 train=3.66805902e-03 val=3.46026372e-03 grad_max=8.124e-04 step_max=nan loss_change=nan dA=0.000e+00 dX*=0.000e+00 best_val=3.46026372e-03

## Optional evaluation with ground truth

If a ground-truth adjacency matrix is available, `evaluate()` computes perturbation AUROC from the scores written by `fit()`.

In [7]:
eval_metrics = model.evaluate(A_true_binary)
print(f"AUROC: {eval_metrics['auroc_perturb']:.3f}")


AUROC: 0.591
